In [7]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import ts_onset_cess as ocd
import pandas as pd
from fapar_def import fapar_read

import warnings
warnings.filterwarnings('ignore')

In [8]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/data/"

### Read in precip 
- do this for whole available ts and then sub-select years in next step for analysis

In [9]:
Y1=2018
Y2=2024

prp_all_list = []
for Y in range(Y1,Y2+1):
    prp = xr.open_mfdataset('/Volumes/blue_wd/chirps_daily/chirps-v2.0.'+str(Y)+'.days_p05.nc')['precip']
    prp = prp.rename({'latitude':'lat','longitude':'lon'})
    prp = prp.sel(lat=slice(-15,-5),lon=slice(12,31),drop=True).load()
    prp_year_list = []
    for m in range(1,13):
        try:
            mp = prp.sel(time=(prp.time.dt.month==m), drop=True)
            #print(mp)
            bins = [mp.time[0].values,mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]
            mp_out = mp.groupby_bins('time', bins,labels=[mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]).mean()
            mp_out = mp_out.rename({'time_bins':'time'})
            #print(mp_out)
            prp_year_list.append(mp_out)
        except:
            print('no month - ',m,' for year - ',Y)
    prp_year = xr.concat(prp_year_list,dim='time')
    prp_all_list.append(prp_year)
    prp.close()
    print('done - ',Y)
prp_all = xr.concat(prp_all_list,dim='time')
prp_all = prp_all.sel(time=slice('2018-07-01','2024-12-31'))
print(prp_all)
    
prp_all.to_netcdf(dataf+'chirps_10day_s.nc',engine='h5netcdf')
        

done -  2018
done -  2019
done -  2020
done -  2021
done -  2022
done -  2023
done -  2024
<xarray.DataArray 'precip' (time: 234, lat: 200, lon: 380)> Size: 71MB
array([[[       nan,        nan,        nan, ...,  0.       ,
          0.       ,  0.       ],
        [       nan,        nan,        nan, ...,  0.       ,
          0.       ,  0.       ],
        [       nan,        nan,        nan, ...,  0.       ,
          0.       ,  0.       ],
        ...,
        [       nan,        nan,  0.       , ...,  0.       ,
          0.       ,  0.       ],
        [       nan,  0.       ,  0.       , ...,  0.       ,
          0.       ,  0.       ],
        [ 0.       ,  0.       ,  0.       , ...,  0.       ,
          0.       ,  0.       ]],

       [[       nan,        nan,        nan, ...,  0.       ,
          0.       ,  0.       ],
        [       nan,        nan,        nan, ...,  0.       ,
          0.       ,  0.       ],
        [       nan,        nan,        nan, ...,  0.  